# ZORE AI — Blender Cloud Worker (Google Colab)

Jalankan Blender di server cloud Colab tanpa komputer pribadi. Sesi Colab gratis bersifat sementara dan tidak berjalan 24/7. Notebook ini menyiapkan Blender, membuat contoh aset 3D, menyimpan `.blend` dan `.glb` ke Google Drive. Setelah alur ini terbukti, worker antrean tugas ZORE AI dapat dihubungkan pada tahap berikutnya.

**Jangan masukkan Supabase service-role key, password, atau token rahasia ke notebook.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/ZORE-AI/Blender'
os.makedirs(WORK_DIR, exist_ok=True)
print('Folder kerja:', WORK_DIR)

In [ ]:
# Pasang Blender pada sesi Colab (butuh beberapa menit)
!apt-get update -qq
!apt-get install -y blender
!blender --version | head -n 2

In [ ]:
# Buat scene contoh sederhana, simpan file .blend dan ekspor .glb
import subprocess, textwrap, os

script = r'''import bpy
from mathutils import Vector
import os

out_dir = os.environ.get('ZORE_OUTPUT_DIR', '/content')
os.makedirs(out_dir, exist_ok=True)
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

# Lantai dan objek contoh
bpy.ops.mesh.primitive_plane_add(size=8, location=(0,0,0))
floor = bpy.context.object
floor.name = 'ZORE_Office_Floor'
mat = bpy.data.materials.new('Warm_Wood_PBR')
mat.diffuse_color = (0.28, 0.16, 0.08, 1)
floor.data.materials.append(mat)

bpy.ops.mesh.primitive_cube_add(size=1.5, location=(0,0,0.8))
cube = bpy.context.object
cube.name = 'ZORE_Asset_Test'
cube_mat = bpy.data.materials.new('ZORE_Brass')
cube_mat.diffuse_color = (0.65, 0.38, 0.12, 1)
cube.data.materials.append(cube_mat)

bpy.ops.object.select_all(action='SELECT')
bpy.ops.wm.save_as_mainfile(filepath=os.path.join(out_dir, 'zore-cloud-test.blend'))
bpy.ops.export_scene.gltf(filepath=os.path.join(out_dir, 'zore-cloud-test.glb'), export_format='GLB')
print('ZORE_EXPORT_OK')
'''

script_path = '/content/zore_colab_job.py'
with open(script_path, 'w') as f:
    f.write(script)

env = os.environ.copy()
env['ZORE_OUTPUT_DIR'] = WORK_DIR
result = subprocess.run(['blender', '-b', '-t', '2', '--python', script_path], env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError('Blender gagal memproses scene')
assert os.path.exists(os.path.join(WORK_DIR, 'zore-cloud-test.blend'))
assert os.path.exists(os.path.join(WORK_DIR, 'zore-cloud-test.glb'))
print('Berhasil: file .blend dan .glb tersimpan di Google Drive.')

## Menggunakan hasil di ZORE AI

1. Jalankan semua sel dari atas ke bawah.
2. Setelah file `.glb` tersimpan di `Google Drive / ZORE-AI / Blender`, unduh file tersebut ke HP jika perlu.
3. Buka halaman preview Babylon ZORE AI dan pilih **Muat model Blender** untuk menguji file `.glb`.
4. Jangan menganggap tugas karyawan otomatis selesai: notebook ini saat ini adalah worker cloud manual/prototipe. Penghubung antrean tugas, otorisasi aman, unggah otomatis ke Storage, dan status hasil perlu tahap integrasi backend tersendiri.

Jika Colab memutus sesi, jalankan ulang sel instalasi dan proses. Simpan hasil penting ke Drive, bukan hanya `/content`.

In [ ]:
# Lihat file hasil
for name in sorted(os.listdir(WORK_DIR)):
    path = os.path.join(WORK_DIR, name)
    print(name, os.path.getsize(path), 'bytes')